# Capítulo 4 — El Lado Oscuro de las Pruebas E2E

> *«Un test que a veces pasa y a veces falla sin que el código haya cambiado no es un test: es ruido estadístico.»*

**Objetivo:** Identificar los problemas más comunes en pruebas E2E — los temidos **flaky tests** —, comprender sus causas, y aplicar técnicas de estabilización. Conocer los antipatrones que convierten una suite E2E en una fuente de dolor en lugar de confianza.

---

## 1. ¿Qué es un flaky test?

Un **flaky test** (test inestable) es un test que:

- **A veces pasa, a veces falla**, sin que el código haya cambiado.
- No reproduce el mismo resultado de forma determinista.
- Hace que los desarrolladores ignoren los fallos de los tests (*«es que el test es flaky, no es un bug real»*).

Los flaky tests son especialmente peligrosos porque **erosionan la confianza** en la suite de pruebas. Si el equipo aprende a ignorar los fallos, las pruebas dejan de servir como red de seguridad.

### La escala del problema

Google reportó en 2016 que el **1.5% de sus tests** (de millones) eran flaky. En un proyecto típico, el porcentaje puede ser mucho mayor. Un estudio de Microsoft (2020) encontró que los desarrolladores **invierten hasta el 40% del tiempo de debugging** en fallos causados por flakiness.

---

## 2. Causas de flakiness en E2E

### 2.1 Condiciones de carrera (Race Conditions)

El test hace clic en un botón antes de que la página haya terminado de cargar:

```python
# ❌ MAL: puede fallar si la página es lenta
page.goto(url)
page.locator("[data-testid='btn-agregar']").click()  # ← la página puede no estar lista

# ✅ BIEN: esperar explícitamente
page.goto(url)
page.wait_for_load_state('networkidle')
page.locator("[data-testid='btn-agregar']").click()
# O mejor aún, usar el auto-wait de Playwright:
page.get_by_test_id('btn-agregar').click()  # espera automáticamente a que sea clickeable
```

### 2.2 Esperas fijas (time.sleep)

```python
# ❌ MAL: espera fija — lento en CI y aún puede fallar
page.click("[data-testid='btn-agregar']")
time.sleep(2)  # ← ¿y si el servidor tarda 3 segundos?
assert page.locator("[data-testid='tarea-titulo']").count() == 1

# ✅ BIEN: esperar por la condición específica
page.click("[data-testid='btn-agregar']")
page.wait_for_selector("[data-testid='tarea-titulo']")  # espera hasta que aparezca
assert page.locator("[data-testid='tarea-titulo']").count() == 1
```

### 2.3 Estado compartido entre tests

```python
# ❌ MAL: test 1 deja datos que afectan al test 2
def test_crear(page, live_server):
    tp = TaskPage(page)
    tp.navegar(live_server)
    tp.agregar_tarea('Tarea de test 1')
    assert tp.cantidad_tareas() == 1

def test_lista_vacia(page, live_server):  # ← asume lista vacía, pero test_crear dejó una
    tp = TaskPage(page)
    tp.navegar(live_server)
    assert tp.lista_esta_vacia()  # FALLA porque test_crear dejó una tarea

# ✅ BIEN: fixture autouse que limpia antes de cada test (ver conftest.py)
```

### 2.4 Dependencia del orden de ejecución

Los tests **nunca deben depender del orden** en que se ejecutan. Si `test_B` necesita el estado que deja `test_A`, ambos tests son inestables.

```python
# ❌ MAL: test_completar asume que test_crear ya se ejecutó
def test_completar(page, live_server):  # ← ¿y si se ejecuta solo?
    tp = TaskPage(page)
    tp.navegar(live_server)
    tp.completar_tarea('Tarea creada en otro test')  # lanza ValueError

# ✅ BIEN: cada test es autosuficiente
def test_completar(page, live_server):
    tp = TaskPage(page)
    tp.navegar(live_server)
    tp.agregar_tarea('Tarea propia de este test')  # crea su propio estado
    tp.completar_tarea('Tarea propia de este test')
    assert tp.tarea_esta_completada('Tarea propia de este test')
```

### 2.5 Selectores frágiles

```python
# ❌ MAL: frágil ante cambios de posición o estructura HTML
page.locator('ul > li:first-child > div > button:nth-child(2)').click()

# ⚠ ACEPTABLE: CSS semántico, mejor que estructural
page.locator('button.btn-complete').click()

# ✅ MEJOR: data-testid estable por diseño
page.locator("[data-testid='btn-completar']").first.click()

# ✅ IDEAL: roles y texto accesibles (resiste cambios de atributos)
page.get_by_role('button', name='Completar').first.click()
```

### 2.6 Dependencias externas no controladas

Los tests E2E que llaman a APIs externas reales (email, pagos, mapas) son inherentemente inestables.

```python
# ✅ Interceptar peticiones externas en Playwright
page.route('**/api.external.com/**', lambda route: route.fulfill(
    status=200, body='{"result": "mocked"}'
))
```

In [ ]:
# Demostración: diagnosticar un test flaky
# Simulamos un escenario donde el orden importa y causa inestabilidad

import sys, os
sys.path.insert(0, os.path.abspath('..'))

# Supongamos que tenemos estos tests sin aislamiento:
test_results = []

def simular_test(nombre, estado_inicial, accion, verificacion):
    try:
        resultado = verificacion(accion(estado_inicial))
        test_results.append((nombre, 'PASA', estado_inicial))
        print(f'✓ {nombre} (estado inicial: {estado_inicial} tareas)')
    except AssertionError as e:
        test_results.append((nombre, 'FALLA', estado_inicial))
        print(f'✗ {nombre} (estado inicial: {estado_inicial} tareas) → {e}')

# Test que asume lista vacía
def test_lista_vacia(estado):
    return estado
def verificar_vacia(estado):
    assert estado == 0, f'Esperaba 0 tareas, encontró {estado}'

# Test que crea una tarea
def test_crear(estado):
    return estado + 1
def verificar_una(estado):
    assert estado == 1, f'Esperaba 1 tarea, encontró {estado}'

print('=== Sin aislamiento (orden 1) ===')
estado = 0
simular_test('test_lista_vacia', estado, test_lista_vacia, verificar_vacia)
estado = test_crear(estado)  # estado = 1
simular_test('test_crear',     estado, test_lista_vacia, verificar_una)

print()
print('=== Sin aislamiento (orden 2 — invertido) ===')
estado = 0
estado = test_crear(estado)  # estado = 1 — se ejecuta primero
simular_test('test_crear',     estado, test_lista_vacia, verificar_una)
simular_test('test_lista_vacia', estado, test_lista_vacia, verificar_vacia)  # FALLA

print()
print('CONCLUSIÓN: El mismo test puede pasar o fallar según el orden.')
print('La solución es limpiar el estado antes de cada test.')

---

## 3. Técnicas de estabilización

### 3.1 Auto-wait de Playwright

Playwright espera automáticamente a que los elementos sean:
- **Visibles** en el DOM
- **Habilitados** (no deshabilitados)
- **Estables** (no animándose)
- **Recibir eventos** (no oscurecidos por otro elemento)

Esto elimina la mayoría de las race conditions sin necesidad de `sleep` ni `wait_for_selector` explícito.

```python
# Playwright espera automáticamente
page.locator("[data-testid='btn-agregar']").click()  # espera hasta poder hacer clic
```

### 3.2 Aserciones con `expect()`

```python
from playwright.sync_api import expect

# expect() reintenta hasta que la condición se cumpla (timeout configurable)
expect(page.locator("[data-testid='tarea-titulo']")).to_have_count(1)
expect(page.locator("[data-testid='badge-completada']")).to_be_visible()
expect(page.locator("[data-testid='msg-lista-vacia']")).to_contain_text('No hay tareas')
```

### 3.3 Interceptar la red

```python
# Controlar respuestas externas para hacer los tests deterministas
page.route('**/api.externa.com/datos', lambda route: route.fulfill(
    status=200,
    content_type='application/json',
    body='{"datos": ["item1", "item2"]}'
))
```

### 3.4 Estrategia de retry en CI/CD

Para tests inevitablemente flaky (ej. integraciones con servicios de terceros), configurar reintentos en la CI:

```ini
# pytest.ini
[pytest]
# Requiere pytest-rerunfailures
# addopts = --reruns 2 --reruns-delay 1
```

> ⚠️ **Cuidado:** El retry es un parche, no una solución. Si un test necesita retry, tiene un problema de diseño.

---

## 4. Antipatrones en pruebas E2E

| Antipatrón | Descripción | Consecuencia |
|-----------|-------------|-------------|
| **Pirámide invertida** | Más E2E que unitarias | Suite lenta, cara, frágil |
| **Tests sin aislamiento** | Comparten estado entre tests | Flakiness por orden de ejecución |
| **Esperas fijas** | `time.sleep(2)` | Lentos y aún inestables |
| **Selectores frágiles** | CSS/XPath estructurales | Fallan ante refactoring de UI |
| **Tests demasiado largos** | Un solo test verifica 20 cosas | Difícil de diagnosticar al fallar |
| **Tests sin Page Object** | Selectores duplicados en todos los tests | Mantenimiento costoso |
| **Datos de producción** | Tests corren contra BD real | Datos contaminados, tests destructivos |
| **Screenshots como aserciones** | Comparar capturas de pantalla pixel a pixel | Rompen ante mínimos cambios de CSS |

---

## 5. E2E en CI/CD — consideraciones prácticas

### Pipeline recomendado

```
Push → [Unitarias] → [Integración] → [E2E smoke] → Deploy → [E2E full]
                                          ↑
                              Solo flujos críticos (rápido)
```

### Configuración de headless en CI

```python
# conftest.py — headless automático en CI
import os

@pytest.fixture
def browser_options():
    headless = os.environ.get('CI', 'false').lower() == 'true'
    return {'headless': headless}
```

### Artefactos de debugging en CI

```python
# Capturar screenshot y traza cuando falla
@pytest.hookimpl(tryfirst=True, hookwrapper=True)
def pytest_runtest_makereport(item, call):
    outcome = yield
    report = outcome.get_result()
    if report.when == 'call' and report.failed:
        page = item.funcargs.get('page')
        if page:
            page.screenshot(path=f'screenshots/{item.name}.png')
```

---

## 6. Reflexión final

1. **Diagnóstico de flakiness**: Describe tres escenarios concretos en el gestor de tareas donde podría surgir un flaky test. Para cada uno, explica la causa y propón la solución.

2. **Pirámide de pruebas**: Si tienes 200 tests unitarios, 50 de integración y 5 E2E en tu proyecto, ¿estás siguiendo la pirámide correctamente? ¿Qué ocurriría si invirtieras esa proporción?

3. **Aislamiento vs. velocidad**: Limpiar la base de datos antes de cada test garantiza aislamiento pero hace los tests más lentos. ¿Qué otras estrategias de aislamiento conoces que sean más eficientes?

4. **CI/CD**: En tu pipeline de CI/CD, ¿ejecutarías los tests E2E en cada push o solo antes de hacer deploy? Justifica tu decisión con argumentos sobre velocidad de feedback vs. cobertura.